### THE CASE OF FLIXBUS: POSITIONING STRATEGY AND PRICE ALLOCATION IN LONG DISTANCE BUS SERVICE
###### Merlin Mirley Moreno Palacios 
###### Date: 04/2026


In [7]:
# Data manipulation
import pandas as pd
import numpy as np
# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

#### 1. Data loading and exploratory inspection

I start by loading the dataset from an Excel source and performing an initial inspection. This step helps verify the structure of the data, detect missing values, and understand the distributions of the main variables before applying any transformations.

To do this, I examine the dataset structure, variable types, and summary statistics.

In [ ]:
# Load dataset
df_final = pd.read_excel("DF.xlsx")

# Initial inspection
print(df_final.info())
display(df_final.describe(include="all"))
display(df_final.head())

#### 2. Fare variables and data cleaning

In the dataset, two different fare measures are constructed in order to capture distinct pricing scenarios observed on the FlixBus platform.

The variable *fare1* represents the price associated with the first available seat for a given query. This variable approximates the lowest fare offered at a given point in time and is therefore used as the main dependent variable in the empirical analysis. It reflects the initial price level faced by consumers when searching for tickets.

In contrast, *fare2* represents an alternative pricing measure that accounts for situations in which the booking request does not necessarily correspond to the first available seat. This occurs particularly in censored observations, where the platform returns valid prices for all seat requests (from 1 up to 42 seats), making it unclear whether the observed price corresponds to the marginal seat or to an adjusted pricing structure.

The distinction between *fare1* and *fare2* is crucial for the econometric analysis, as it allows the comparison between censored and non-censored data and provides robustness in the estimation of pricing dynamics under different availability scenarios.

Additionally, zero values in both fare variables are treated as invalid observations rather than true prices. To avoid introducing bias into the analysis, these values are replaced with missing values (`NaN`).

In [9]:
df_final["fare1"] = df_final["fare1"].replace(0, np.nan)
df_final["fare2"] = df_final["fare2"].replace(0, np.nan)

#### 3. Panel structure and identifier construction

In order to implement the econometric model, the dataset is structured as panel data. Each observation corresponds to a specific route and booking date, allowing the analysis of fare dynamics over time.

To define the panel dimension, a unique identifier is constructed by combining the route identifier (*routeID*) with the day of departure (*ddate*). This approach allows grouping observations that belong to the same route and departure date, ensuring consistency in the analysis of fare evolution.

This structure is consistent with the model specification, where fare dynamics are analyzed as a function of both capacity (available seats) and time to departure. By organizing the data in this way, it becomes possible to control for unobserved heterogeneity at the route level and track pricing behavior across different booking periods.

In [10]:
# Ensure date format
df_final["ddate"] = pd.to_datetime(df_final["ddate"])

# Create unique panel identifier (route + full date)
df_final["id"] = (
    df_final["routeID"].astype(str) + "_" + df_final["ddate"].dt.strftime("%Y%m%d")
)

#### 4. Capacity-based feature engineering

A key dimension of the analysis is the role of capacity in fare determination. Following the empirical framework of the study, available seats are used as a central explanatory variable to understand how prices evolve as seat availability changes over time.

To complement this information, I construct the variable *capacity_reverse*, defined as the minimum number of available seats observed within each route–departure combination. This variable summarizes the lowest capacity level reached for a given trip and helps characterize the degree of seat scarcity within that booking window.

This transformation is consistent with the underlying pricing logic discussed in the paper, where fare adjustments are expected to respond to remaining capacity and demand conditions.

In [11]:
# Create a working copy of the dataset
df_final1 = df_final.copy()

# Minimum number of available seats within each route-departure group
df_final1["capacity_reverse"] = df_final1.groupby("id")["sleft"].transform("min")

#### 5. Maximum fare within each route–departure combination

To further characterize pricing behavior, I construct the variable *topfare*, defined as the maximum fare observed within each route–departure combination.

This variable captures the upper bound of the fare distribution for a given trip and provides additional information on the extent of price variation within the same service. In the context of revenue management, this is useful for identifying how high fares can rise under specific capacity and booking conditions.

In [12]:
# Maximum observed fare within each route-departure group
df_final1["topfare"] = df_final1.groupby("id")["fare"].transform("max")

#### 6. Fare distribution within each query

To describe the fare structure returned by each search, I compute summary statistics at the query level. Specifically, I construct the mean, median, and maximum fare observed within each *queryID*.

These measures help capture the distribution of prices associated with a given query and provide additional context for understanding how fares vary across booking conditions. They are useful descriptive indicators of the pricing environment observed on the platform.

In [13]:
# Fare summary statistics within each query
df_final1["mean_fare"] = df_final1.groupby("queryID")["fare"].transform("mean")
df_final1["median_fare"] = df_final1.groupby("queryID")["fare"].transform("median")
df_final1["max_fare"] = df_final1.groupby("queryID")["fare"].transform("max")

#### 7. Group identifiers for routes and departure dates

To facilitate the analysis and modeling process, additional grouping variables are constructed.

First, a variable (*book*) is created to identify each departure date. This allows grouping observations that correspond to the same travel day and helps organize the temporal structure of the dataset.

Second, a route-level identifier (*route_group*) is defined to distinguish between different origin–destination pairs. This is particularly useful for capturing route-specific characteristics and enables a clearer comparison of pricing patterns across different routes.

These grouping variables support the panel data structure and improve the interpretability of the analysis.

In [14]:
# Create numeric group identifier by departure date
df_final1["book"] = df_final1.groupby("ddate").ngroup() + 1

# Create numeric group identifier by route
df_final1["route_group"] = df_final1.groupby("route").ngroup() + 1

#### 8. Logarithmic transformation of fare variables

Following the empirical specification presented in the study, the fare variables are transformed using the natural logarithm. In particular, the logarithm of *fare1* is used as the main dependent variable in the econometric models, as described in the model section of the paper.

This transformation serves two purposes. First, it makes the interpretation of the coefficients more meaningful in relative terms, since estimated effects can be read approximately as percentage changes. Second, it reduces the influence of extreme fare values and helps stabilize the distribution of the dependent variable.

The same transformation is also applied to *fare2* in order to compare results under the alternative pricing scenario associated with censored observations.

In [15]:
# Log-transformed fare variables
df_final1["lnfare1"] = np.where(df_final1["fare1"] > 0, np.log(df_final1["fare1"]), np.nan)
df_final1["lnfare2"] = np.where(df_final1["fare2"] > 0, np.log(df_final1["fare2"]), np.nan)

#### 9. Temporal variables for departure and booking behavior

Since the analysis focuses on fare dynamics over time, I construct a set of temporal variables based on both the departure date and the booking date.

First, I generate the day of the week for the departure date (*dow_departure*) and for the booking date (*dowb*). These variables help describe the timing of trips and searches across the week.

Second, I define two binary indicators: *weekend_departure* and *weekend_book*. These variables identify whether the departure or booking occurred on Friday, Saturday, or Sunday.

This distinction is relevant for the empirical analysis because booking behavior may differ systematically on weekends, and the variable *weekend_book* is later used as an instrumental variable to address the potential endogeneity of available seats. This is consistent with the discussion in the paper, where weekend booking is tested as a relevant instrument in the pricing model.

In [17]:
# Ensure booking date is in datetime format
df_final1["qdata"] = pd.to_datetime(df_final1["qdata"])

# Day of week for departure date: Monday=0, ..., Sunday=6
df_final1["dow_departure"] = df_final1["ddate"].dt.dayofweek

# Weekend departure: Friday, Saturday, Sunday = 1
df_final1["weekend_departure"] = np.where(
    df_final1["ddate"].dt.day_name().isin(["Friday", "Saturday", "Sunday"]),
    1,
    0
)

# Day of week for booking date: Monday=0, ..., Sunday=6
df_final1["dowb"] = df_final1["qdata"].dt.dayofweek

# Weekend booking: Friday, Saturday, Sunday = 1
df_final1["weekend_book"] = np.where(
    df_final1["qdata"].dt.day_name().isin(["Friday", "Saturday", "Sunday"]),
    1,
    0
)

#### 10. Route-level market characteristics

To account for structural differences across routes, I construct a set of market-level variables based on the origin city, destination city, and route characteristics.

First, I identify the origin and destination of each observation and assign corresponding measures of population and GDP. These variables are intended to proxy the economic scale of the markets connected by each route.

Second, I define the main direct travel substitute available for each route, distinguishing between train, airplane, and no direct substitute. This is particularly relevant in the context of the study, since the dataset includes routes with different competitive environments. As described in the paper, some routes face direct intermodal competition, while others are operated exclusively by FlixBus. :contentReference[oaicite:1]{index=1}

I also include route distance and construct aggregate indicators of market size by taking the geometric mean of origin and destination population and GDP. Finally, I create the variable *flixbus_only*, which identifies routes without a direct substitute. This variable is useful for examining whether pricing differs when FlixBus faces weaker competitive pressure.

In [18]:
# Origin and destination variables
df_final1["origin"] = df_final1["source"]
df_final1["destination"] = df_final1["target"]

# Population and GDP mappings for origin cities
pop_origin_map = {
    "munich": 16.043,
    "tuebingen": 0.892,
    "hanover": 5.359,
    "stuttgart": 6.000
}

gdp_origin_map = {
    "munich": 86.529,
    "tuebingen": 40.100,
    "hanover": 87.513,
    "stuttgart": 57.100
}

# Population and GDP mappings for destination cities
pop_destination_map = {
    "nuremberg": 5.225,
    "freiburg": 2.330,
    "bremen": 5.751,
    "milan": 13.620
}

gdp_destination_map = {
    "nuremberg": 62.997,
    "freiburg": 35.300,
    "bremen": 54.826,
    "milan": 49.500
}

# Assign route-level market characteristics
df_final1["pop_o"] = df_final1["origin"].map(pop_origin_map)
df_final1["gdp_o"] = df_final1["origin"].map(gdp_origin_map)
df_final1["pop_d"] = df_final1["destination"].map(pop_destination_map)
df_final1["gdp_d"] = df_final1["destination"].map(gdp_destination_map)

# Travel substitute by route
travel_sub_map = {
    "munich-nuremberg": "train",
    "stuttgart-milan": "airplane",
    "tuebingen-freiburg": "none",
    "hanover-bremen": "train"
}
df_final1["travel_sub"] = df_final1["route"].map(travel_sub_map)

# Distance by route
distance_map = {
    "munich-nuremberg": 169,
    "stuttgart-milan": 505,
    "tuebingen-freiburg": 159,
    "hanover-bremen": 128
}
df_final1["dist"] = df_final1["route"].map(distance_map)

# Geometric means
df_final1["pop"] = np.sqrt(df_final1["pop_o"] * df_final1["pop_d"])
df_final1["gdp"] = np.sqrt(df_final1["gdp_o"] * df_final1["gdp_d"])

# Indicator for routes operated without a direct substitute
df_final1["flixbus_only"] = np.where(df_final1["travel_sub"] == "none", 1, 0)

#### 11. Time-to-departure grouping

To capture the temporal dimension of pricing, I transform the variable *bookingday* into a grouped categorical measure called *day_departured*. This variable classifies each observation according to how many days before departure the query was made.

Following the empirical specification described in the paper, the time dimension is represented through eight booking windows rather than individual days. This grouping makes it possible to compare fare behavior across different stages of the booking horizon while keeping the model.

The resulting categories distinguish between very late bookings, intermediate booking periods, and early bookings. This is important for testing whether fares increase as departure approaches or whether operators reduce prices when there is still substantial unsold capacity.

In [19]:
# Group booking days into departure windows
df_final1["day_departured"] = np.select(
    [
        df_final1["bookingday"] == 1,
        df_final1["bookingday"] == 2,
        df_final1["bookingday"] == 3,
        df_final1["bookingday"].between(4, 5),
        df_final1["bookingday"].between(6, 7),
        df_final1["bookingday"].between(8, 14),
        df_final1["bookingday"].between(15, 21),
        df_final1["bookingday"].between(22, 28),
    ],
    [1, 2, 3, 4, 5, 6, 7, 8],
    default=np.nan
)

#### 12. Sample construction by fare definition

To estimate the pricing models under different fare specifications, I split the dataset into two analytical samples.

The first sample includes observations with valid values for *fare1*, which represents the main price measure used in the empirical analysis. The second sample includes observations with valid values for *fare2*, which captures the alternative pricing scenario associated with censored observations.

This distinction is consistent with the data collection strategy described in the paper, where the availability of fare information differs depending on whether the platform reveals the exact first-seat price or only a censored pricing outcome.

In [21]:
# Filter datasets for each fare specification
data_r = df_final1[df_final1["fare1"].notna()].copy()
data_r2 = df_final1[df_final1["fare2"].notna()].copy()

#### 13. Panel data preparation

After constructing the analytical samples, I prepare the data for panel estimation. Each dataset is organized using a two-dimensional index defined by the route–departure identifier and the booking date. This structure makes it possible to follow the evolution of fares for the same trip across different booking moments.

I also ensure that the main variables have the appropriate data types for estimation. In particular, the grouped time-to-departure variable (*day_departured*) is explicitly treated as categorical, with the 22–28 days interval set as the reference category. This follows the empirical specification described in the paper, where early booking periods serve as the baseline for comparison. :contentReference[oaicite:1]{index=1}

This preparation step is necessary for estimating fixed-effects and instrumental-variable models in a consistent way.

In [33]:
# Prepare first panel dataset
pdata = data_r.copy()
pdata["qdata"] = pd.to_datetime(pdata["qdata"])
pdata["bookingday"] = pdata["bookingday"].astype("category")
pdata["seat"] = pd.to_numeric(pdata["seat"], errors="coerce")
pdata["weekend_departure"] = pd.to_numeric(pdata["weekend_departure"], errors="coerce")
pdata["lnfare1"] = pd.to_numeric(pdata["lnfare1"], errors="coerce")
pdata["book"] = pdata["book"].astype("category")

# Set category 8 as the reference level for day_departured
pdata["day_departured"] = pd.Categorical(
    pdata["day_departured"],
    categories=[8, 1, 2, 3, 4, 5, 6, 7],
    ordered=False
)

# Set panel index
pdata = pdata.set_index(["id", "qdata"]).sort_index()


# Prepare second panel dataset
pdata2 = data_r2.copy()
pdata2["qdata"] = pd.to_datetime(pdata2["qdata"])
pdata2["bookingday"] = pdata2["bookingday"].astype("category")
pdata2["seat"] = pd.to_numeric(pdata2["seat"], errors="coerce")
pdata2["weekend_departure"] = pd.to_numeric(pdata2["weekend_departure"], errors="coerce")
pdata2["lnfare2"] = pd.to_numeric(pdata2["lnfare2"], errors="coerce")
pdata2["book"] = pdata2["book"].astype("category")

# Set category 8 as the reference level for day_departured
pdata2["day_departured"] = pd.Categorical(
    pdata2["day_departured"],
    categories=[8, 1, 2, 3, 4, 5, 6, 7],
    ordered=False
)

# Set panel index
pdata2 = pdata2.set_index(["id", "qdata"]).sort_index()

#### 14. Fixed-effects model specification

To estimate the relationship between seat availability, booking timing, and fares, I begin with fixed-effects panel models. This specification is consistent with the empirical framework described in the paper, where fare dynamics are analyzed within route–departure units over time. :contentReference[oaicite:0]{index=0}

The grouped time-to-departure variable is introduced through dummy variables, using the 22–28 days category as the reference level. This makes it possible to compare how seat availability and fares evolve as the departure date approaches.

In [ ]:
!pip install linearmodels

In [34]:
from linearmodels.panel import PanelOLS
import pandas as pd

# Reset index to create dummies
pdata_model = pdata.reset_index().copy()

# Create dummy variables for day_departured
day_dummies = pd.get_dummies(
    pdata_model["day_departured"],
    prefix="day",
    drop_first=False
)

# Drop reference category (day_8)
day_dummies = day_dummies.drop(columns=["day_8"], errors="ignore")

# Merge dummies into dataset
pdata_model = pd.concat([pdata_model, day_dummies], axis=1)

# Restore panel index
pdata_model = pdata_model.set_index(["id", "qdata"]).sort_index()

# Keep only the real dummy columns
day_cols = [col for col in pdata_model.columns if col.startswith("day_") and col != "day_departured"]
day_cols = sorted(day_cols)

print(day_cols)

['day_1', 'day_2', 'day_3', 'day_4', 'day_5', 'day_6', 'day_7']


#### Model 1: Available seats and time to departure

The first model estimates how seat availability evolves across booking periods. The dependent variable is the number of available seats (*sleft*), while the explanatory variables are the grouped booking windows.

This model helps identify whether seat availability declines as departure approaches, which is a necessary condition for the pricing mechanism analyzed in the paper. A negative relationship between later booking periods and available seats would be consistent with progressive seat depletion over time.

In [35]:
# Model 1: sleft ~ day_departured (fixed effects)

exog1 = pdata_model[day_cols].astype(float)

model1_simple_sleft = PanelOLS(
    dependent=pdata_model["sleft"].astype(float),
    exog=exog1,
    entity_effects=True
).fit(cov_type="robust")

print(model1_simple_sleft.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:                  sleft   R-squared:                        0.4630
Estimator:                   PanelOLS   R-squared (Between):             -0.0590
No. Observations:                3020   R-squared (Within):               0.4630
Date:                Fri, Apr 10 2026   R-squared (Overall):             -0.0941
Time:                        18:42:25   Log-likelihood                   -7313.3
Cov. Estimator:                Robust                                           
                                        F-statistic:                      349.31
Entities:                         177   P-value                           0.0000
Avg Obs:                       17.062   Distribution:                  F(7,2836)
Min Obs:                       1.0000                                           
Max Obs:                       28.000   F-statistic (robust):             148.45
                            

In [37]:
# Model 2: lnfare1 ~ sleft + day_departured (fixed effects)

exog2 = pdata_model[["sleft"] + day_cols].astype(float)

model2_simple_lnfare1 = PanelOLS(
    dependent=pdata_model["lnfare1"].astype(float),
    exog=exog2,
    entity_effects=True
).fit(cov_type="robust")

print(model2_simple_lnfare1.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:                lnfare1   R-squared:                        0.2282
Estimator:                   PanelOLS   R-squared (Between):             -0.4318
No. Observations:                3020   R-squared (Within):               0.2282
Date:                Fri, Apr 10 2026   R-squared (Overall):             -0.4055
Time:                        18:44:23   Log-likelihood                    1713.7
Cov. Estimator:                Robust                                           
                                        F-statistic:                      104.77
Entities:                         177   P-value                           0.0000
Avg Obs:                       17.062   Distribution:                  F(8,2835)
Min Obs:                       1.0000                                           
Max Obs:                       28.000   F-statistic (robust):             80.173
                            

In [38]:
model2_simple_lnfare1.params

sleft   -0.014651
day_1    0.111873
day_2    0.113995
day_3    0.077341
day_4    0.001148
day_5    0.005430
day_6    0.032352
day_7    0.006449
Name: parameter, dtype: float64

In [39]:
# Full coefficient table for Model 1
results_model1 = pd.DataFrame({
    "Coefficient": model1_simple_sleft.params,
    "P-value": model1_simple_sleft.pvalues
})
display(results_model1)

# Full coefficient table for Model 2
results_model2 = pd.DataFrame({
    "Coefficient": model2_simple_lnfare1.params,
    "P-value": model2_simple_lnfare1.pvalues
})
display(results_model2)

,Coefficient,P-value
day_1,-11.159070,0.000000e+00
day_2,-7.988179,0.000000e+00
day_3,-6.228189,0.000000e+00
day_4,-5.071405,0.000000e+00
day_5,-3.477945,0.000000e+00
day_6,-1.813735,0.000000e+00
day_7,-0.686109,3.509292e-08


,Coefficient,P-value
sleft,-0.014651,0.000000e+00
day_1,0.111873,1.648667e-06
day_2,0.113995,6.833933e-11
day_3,0.077341,2.108442e-06
day_4,0.001148,9.590927e-01
day_5,0.005430,8.032563e-01
day_6,0.032352,7.164053e-09
day_7,0.006449,2.024932e-01


In [40]:
# Reset index to create dummies for pdata2
pdata2_model = pdata2.reset_index().copy()

# Create dummy variables for day_departured
day_dummies2 = pd.get_dummies(
    pdata2_model["day_departured"],
    prefix="day",
    drop_first=False
)

# Drop reference category (day_8)
day_dummies2 = day_dummies2.drop(columns=["day_8"], errors="ignore")

# Merge dummies into dataset
pdata2_model = pd.concat([pdata2_model, day_dummies2], axis=1)

# Restore panel index
pdata2_model = pdata2_model.set_index(["id", "qdata"]).sort_index()

# Keep only the real dummy columns
day_cols2 = [col for col in pdata2_model.columns if col.startswith("day_") and col != "day_departured"]
day_cols2 = sorted(day_cols2)

print(day_cols2)

['day_1', 'day_2', 'day_3', 'day_4', 'day_5', 'day_6', 'day_7']


#### 15. Instrumental-variable estimation and model comparison

After estimating the baseline fixed-effects models, I implement an instrumental-variable specification to address the potential endogeneity of available seats. Following the logic discussed in the paper, *weekend_book* is used as an instrument for *sleft*, while the booking-window dummies remain as controls. :contentReference[oaicite:1]{index=1}

To facilitate comparison, I summarize the baseline fixed-effects models and the IV model in a single results table. This allows a direct assessment of how the estimated effect of seat availability changes once endogeneity is taken into account.

In [41]:
from linearmodels.iv import IV2SLS

# Prepare data for IV model
pdata_iv = pdata_model.reset_index().copy()

# Model 3: IV regression
model3_instrument = IV2SLS.from_formula(
    "lnfare1 ~ 1 + " + " + ".join(day_cols) + " + [sleft ~ weekend_book]",
    data=pdata_iv
).fit(cov_type="robust")

print(model3_instrument.summary)

                          IV-2SLS Estimation Summary                          
Dep. Variable:                lnfare1   R-squared:                     -0.0611
Estimator:                    IV-2SLS   Adj. R-squared:                -0.0639
No. Observations:                3020   F-statistic:                    21.096
Date:                Sat, Apr 11 2026   P-value (F-stat)                0.0069
Time:                        15:14:13   Distribution:                  chi2(8)
Cov. Estimator:                robust                                         
                                                                              
                             Parameter Estimates                              
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
Intercept      6.4482     3.0883     2.0880     0.0368      0.3953      12.501
day_1         -0.6974     0.8326    -0.8376     0.40

In [42]:
# Table 1: combine coefficients and p-values from the three models
table1 = pd.concat(
    [
        pd.DataFrame({
            "Model 1: Available seats-FE": model1_simple_sleft.params,
            "Model 1 p-value": model1_simple_sleft.pvalues
        }),
        pd.DataFrame({
            "Model 2: Log(Fare)-FE": model2_simple_lnfare1.params,
            "Model 2 p-value": model2_simple_lnfare1.pvalues
        }),
        pd.DataFrame({
            "Model 3: Log(Fare)-IV": model3_instrument.params,
            "Model 3 p-value": model3_instrument.pvalues
        })
    ],
    axis=1
)

display(table1)

,Model 1: Available seats-FE,Model 1 p-value,Model 2: Log(Fare)-FE,Model 2 p-value,Model 3: Log(Fare)-IV,Model 3 p-value
day_1,-11.159070,0.000000e+00,0.111873,1.648667e-06,-0.697408,0.402230
day_2,-7.988179,0.000000e+00,0.113995,6.833933e-11,-0.465527,0.439086
day_3,-6.228189,0.000000e+00,0.077341,2.108442e-06,-0.381643,0.415595
day_4,-5.071405,0.000000e+00,0.001148,9.590927e-01,-0.372029,0.320232
day_5,-3.477945,0.000000e+00,0.005430,8.032563e-01,-0.248514,0.334022
day_6,-1.813735,0.000000e+00,0.032352,7.164053e-09,-0.101115,0.445764
day_7,-0.686109,3.509292e-08,0.006449,2.024932e-01,-0.038403,0.451996
sleft,NaN,NaN,-0.014651,0.000000e+00,-0.088158,0.239622
Intercept,NaN,NaN,NaN,NaN,6.448238,0.036802


#### 16. Instrumental-variable model with censored data

To assess the robustness of the results, I estimate the instrumental-variable model using the alternative fare definition (*lnfare2*), which reflects censored observations.

This specification allows comparing the pricing dynamics obtained under non-censored and censored scenarios. As discussed in the paper, the presence of censored data may affect the interpretation of the fare variable, since it is not always possible to identify the exact first-seat price.

By estimating the same model with *lnfare2*, I evaluate whether the relationship between seat availability, booking timing, and fares remains consistent across different data conditions.

In [44]:
# Prepare dataset for IV model (fare2)
pdata2_iv = pdata2_model.reset_index().copy()

In [45]:
from linearmodels.iv import IV2SLS

model4_instrument = IV2SLS.from_formula(
    "lnfare2 ~ 1 + " + " + ".join(day_cols2) + " + [sleft ~ weekend_book]",
    data=pdata2_iv
).fit(cov_type="robust")

print(model4_instrument.summary)

                          IV-2SLS Estimation Summary                          
Dep. Variable:                lnfare2   R-squared:                      0.1189
Estimator:                    IV-2SLS   Adj. R-squared:                 0.1116
No. Observations:                 979   F-statistic:                    16.535
Date:                Sat, Apr 11 2026   P-value (F-stat)                0.0353
Time:                        15:19:41   Distribution:                  chi2(8)
Cov. Estimator:                robust                                         
                                                                              
                             Parameter Estimates                              
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
Intercept      3.6025     4.0862     0.8816     0.3780     -4.4064      11.611
day_1         -0.1457     0.9782    -0.1489     0.88

#### 17. Conclusion

This project analyzes pricing dynamics on the FlixBus platform using web-scraped data and a structured data analysis pipeline.

The results provide clear evidence of dynamic pricing behavior. Prices vary systematically with the number of days to departure, showing non-linear patterns as the departure date approaches. This suggests that ticket prices are actively adjusted over time rather than remaining constant.

Additionally, substantial heterogeneity across routes is observed. Some routes exhibit higher average prices and greater price dispersion, indicating that pricing strategies may differ depending on route-specific characteristics such as demand conditions or competition.

The analysis also explores the relationship between travel characteristics and pricing, showing that longer trips tend to be associated with higher fares, although this relationship is not strictly linear.

This project is supported by an accompanying reference document that explains the underlying model, as well as an academic paper that served as a methodological guide. These materials provide the theoretical and empirical foundation for the analysis conducted here.

As a robustness check, the results were compared with equivalent estimations implemented in R. The findings for the primary price variable are fully consistent across both environments. For the alternative price specification, while the exact coefficients differ, the estimated signs remain consistent, supporting the overall interpretation of the results.

Overall, this project demonstrates how web-scraped data can be transformed into meaningful insights through careful data cleaning, feature engineering, and exploratory analysis. The approach used here can be extended to more advanced econometric models to further investigate pricing strategies in transportation markets.